# Lab5.5 — Benchmark CPU vs FPGA și speed-up


Comparăm direct OpenCV pe ARM cu acceleratorul Resize din PL, folosind aceeași imagine și aceeași rezoluție de ieșire.


Pregătim Overlay-ul Resize.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

OVERLAY_DIR = Path("overlay")
OVERLAY_DIR.mkdir(exist_ok=True)
LOCAL_BIT = OVERLAY_DIR / "resizer.bit"
LOCAL_HWH = OVERLAY_DIR / "resizer.hwh"

def find_resizer_overlay():
    if LOCAL_BIT.exists() and LOCAL_HWH.exists():
        return LOCAL_BIT
    try:
        import pynq_helloworld
        pkg = Path(pynq_helloworld.__file__).resolve().parent / "notebooks" / "edge"
        pkg_bit = pkg / "resizer.bit"
        pkg_hwh = pkg / "resizer.hwh"
        if pkg_bit.exists() and pkg_hwh.exists():
            return pkg_bit
    except ImportError:
        pass
    urls = {
        LOCAL_BIT: "https://www.xilinx.com/bin/public/openDownload?filename=pynqhelloworld.resizer.pynqz2.bit",
        LOCAL_HWH: "https://www.xilinx.com/bin/public/openDownload?filename=pynqhelloworld.resizer.pynqz2.hwh",
    }
    for path, url in urls.items():
        if not path.exists():
            print(f"Downloading {path.name} ...")
            urlretrieve(url, path)
    return LOCAL_BIT

RESIZER_BIT = find_resizer_overlay()
print("Overlay:", RESIZER_BIT)

Localizăm imaginea de test comună.


In [ ]:
from pathlib import Path

IMAGE_PATH = Path("../Lab4/images/fruits.jpg")
if not IMAGE_PATH.exists():
    raise FileNotFoundError("Nu a fost găsită imaginea ../Lab4/images/fruits.jpg")
print("Image:", IMAGE_PATH)

Inițializăm imaginea, acceleratorul, DMA-ul și bufferele.


In [ ]:
import cv2
import numpy as np
import time
from pynq import Overlay, allocate

image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
old_h, old_w, _ = image_bgr.shape
new_w, new_h = 320, 240

overlay = Overlay(str(RESIZER_BIT))
dma = overlay.axi_dma_0
resizer = overlay.resize_accel_0
in_buffer = allocate(shape=image_bgr.shape, dtype=np.uint8)
out_buffer = allocate(shape=(new_h, new_w, 3), dtype=np.uint8)
np.copyto(in_buffer, image_bgr)

resizer.register_map.src_rows = old_h
resizer.register_map.src_cols = old_w
resizer.register_map.dst_rows = new_h
resizer.register_map.dst_cols = new_w

def run_kernel():
    dma.sendchannel.transfer(in_buffer)
    dma.recvchannel.transfer(out_buffer)
    resizer.write(0x00, 0x81)
    dma.sendchannel.wait()
    dma.recvchannel.wait()

Executăm câte o rulare de încălzire pentru CPU și PL înaintea măsurătorilor.


In [ ]:
_ = cv2.resize(image_bgr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
run_kernel()
print("Warm-up completed.")

Măsurăm de mai multe ori timpul OpenCV și timpul DMA+accelerator pentru a obține valori medii comparabile.


In [ ]:
REPEATS = 50
cpu_times = []
pl_times = []

for _ in range(REPEATS):
    t0 = time.perf_counter()
    _ = cv2.resize(image_bgr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    cpu_times.append(time.perf_counter() - t0)

for _ in range(REPEATS):
    t0 = time.perf_counter()
    run_kernel()
    pl_times.append(time.perf_counter() - t0)

t_cpu = np.mean(cpu_times)
t_pl = np.mean(pl_times)
speedup = t_cpu / t_pl

print(f"CPU OpenCV : {t_cpu*1e3:8.3f} ms  -> {1/t_cpu:8.2f} FPS")
print(f"PL + DMA   : {t_pl*1e3:8.3f} ms  -> {1/t_pl:8.2f} FPS")
print(f"Speed-up   : {speedup:8.3f} x")

Adăugăm și costul copierii datelor pentru a vedea dacă avantajul acceleratorului se păstrează la nivel end-to-end.


In [ ]:
e2e_times = []

for _ in range(REPEATS):
    t0 = time.perf_counter()
    np.copyto(in_buffer, image_bgr)
    run_kernel()
    _ = np.array(out_buffer)
    e2e_times.append(time.perf_counter() - t0)

t_e2e = np.mean(e2e_times)
print(f"PL end-to-end: {t_e2e*1e3:8.3f} ms  -> {1/t_e2e:8.2f} FPS")
print(f"CPU / E2E speed-up: {t_cpu/t_e2e:8.3f} x")

## Concluzie

Comparația importantă nu este doar `CPU vs nucleu hardware`, ci și `CPU vs întregul traseu de date`. În aplicațiile Edge, transferurile și copiile de memorie pot reduce sau chiar elimina avantajul unui accelerator foarte rapid.


## Exercițiu

Repetați benchmark-ul pentru o altă rezoluție de ieșire, de exemplu **256 × 240**, și comparați speed-up-ul.


Modificați dimensiunile de ieșire, reconfigurați bufferele și repetați măsurătorile.


In [ ]:
# Scrieți aici soluția


Eliberăm bufferele utilizate pentru benchmark.


In [ ]:
del in_buffer, out_buffer


## Întrebări

1. Este acceleratorul mai rapid decât OpenCV în măsurarea `PL + DMA`?
2. Se păstrează avantajul la nivel end-to-end?
3. De ce speed-up-ul depinde de dimensiunea imaginii?
4. Ce concluzie tragem pentru proiectarea unui sistem Edge?
